# 🏥 Asistente Virtual Clínico (Chatbot de Triaje RAG con Gemini AI)

**Autor:** Erick Guardo, Einer plaza
**Asignatura:** Inteligencia Artificial Avanzada
**Profesor:** Carlos Carrascal Avendaño

Este cuaderno implementa un sistema **RAG Completo (Retrieval-Augmented Generation)**.
1. **Búsqueda (Retrieval):** Usa *ClinicalBERT* para entender síntomas y buscar el protocolo médico correcto.
2. **Base de Datos:** Simula consultar MongoDB para ver disponibilidad de médicos.
3. **Generación (Generation):** Usa la IA Generativa de **Google Gemini** para redactar una respuesta empática y natural basada en el protocolo encontrado.

In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS
# Hemos actualizado a la nueva librería oficial de Google (google-genai)
!pip install transformers torch scikit-learn pymongo google-genai -q
print("Librerías instaladas correctamente.")

In [ ]:
# 2. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN DE GEMINI
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
import time
import getpass

print("Por favor, ingresa tu API Key de Google Gemini:")
gemini_api_key = getpass.getpass('API Key: ')
cliente_gemini = genai.Client(api_key=gemini_api_key)

# --- DETECCIÓN AUTOMÁTICA DEL MODELO CORRECTO ---
modelo_disponible = 'gemini-1.5-flash' # fallback
try:
    print("\nBuscando modelos compatibles con tu cuenta...")
    modelos = list(cliente_gemini.models.list())
    nombres = [m.name for m in modelos]
    
    if nombres:
        # Buscamos el mejor modelo disponible para tu cuenta (ej. gemini-2.0, flash-8b, etc.)
        for n in nombres:
            if 'flash' in n or 'pro' in n:
                modelo_disponible = n.replace('models/', '')
                break
        else:
            modelo_disponible = nombres[0].replace('models/', '')
            
    print(f"✅ ¡Éxito! Modelo asignado automáticamente: {modelo_disponible}")
except Exception as e:
    print("\n⚠️ Error al listar modelos (revisa si tu API key tiene restricciones). Usando modelo por defecto.")

# Variable global para el generador
global_modelo_gemini = modelo_disponible


## Arquitectura RAG: Fase 1 - Recuperador Clínico (ClinicalBERT)

In [ ]:
class ClinicalRetriever:
    def __init__(self):
        print("Cargando ClinicalBERT...")
        self.tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        
        self.knowledge_base = [
            "Protocolo Infarto: Dolor opresivo en el pecho, irradiado al brazo izquierdo, sudoración. Acción: Derivar a URGENCIAS INMEDIATAMENTE (Código Rojo).",
            "Protocolo Resfriado: Tos leve, congestión nasal, sin fiebre alta. Acción: Reposo en casa y agendar cita rutinaria (Código Verde).",
            "Protocolo Ansiedad: Palpitaciones, ahogo sin causa física aparente. Acción: Ejercicios de respiración y agendar cita por psicología (Código Amarillo)."
        ]
        self.kb_embeddings = self._compute_embeddings(self.knowledge_base)

    def _compute_embeddings(self, texts):
        inputs = self.tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
        with torch.no_grad():
            outputs = self.model(**inputs)
        return F.normalize(outputs.last_hidden_state.mean(dim=1), p=2, dim=1)

    def retrieve(self, query):
        query_emb = self._compute_embeddings([query])
        sims = cosine_similarity(query_emb.numpy(), self.kb_embeddings.numpy())[0]
        best_idx = np.argmax(sims)
        if sims[best_idx] < 0.3:
            return "Protocolo no encontrado. El paciente presenta síntomas no tipificados."
        return self.knowledge_base[best_idx]

retriever = ClinicalRetriever()
print("Modelo Retriever listo.")

## Simulación de Base de Datos Extendida (Especialidades y Médicos)

In [ ]:
class DatabaseMock:
    def __init__(self):
        self.medicos = {
            "1": {"nombre_esp": "Medicina General", "doctores": [{"nombre": "Fernando Redondo", "disponible": True}]},
            "2": {"nombre_esp": "Cardiología", "doctores": [{"nombre": "Dr. Perez", "disponible": False}, {"nombre": "Dr. Martinez", "disponible": True}]},
            "3": {"nombre_esp": "Psicología", "doctores": [{"nombre": "Dra. Ruiz", "disponible": True}]},
            "4": {"nombre_esp": "Fisioterapia", "doctores": [{"nombre": "Dr. Silva", "disponible": True}]}
        }
    
    def obtener_especialidades(self):
        return self.medicos
        
    def buscar_medico_disponible(self, nombre_esp):
        # Para compatibilidad con el sistema RAG original
        for key, esp in self.medicos.items():
            if esp["nombre_esp"] == nombre_esp:
                for doc in esp["doctores"]:
                    if doc["disponible"]:
                        return doc["nombre"]
        return "Ninguno"

db = DatabaseMock()

## Arquitectura RAG: Fase 2 - Generador (Google Gemini AI)

In [ ]:
def generar_respuesta_gemini(sintomas, protocolo, disponibilidad_medicos):
    prompt = f"""
    Eres el asistente virtual médico de una clínica. Habla de forma empática y profesional, en máximo 2 párrafos cortos.
    
    El paciente ha reportado estos síntomas: '{sintomas}'
    
    Nuestra inteligencia artificial médica determinó este protocolo a seguir: '{protocolo}'
    
    Estado de los médicos en la base de datos: {disponibilidad_medicos}
    
    Instrucciones:
    1. Si es Código Rojo (urgencias), advierte al paciente con firmeza pero sin pánico que debe ir al hospital más cercano porque los cardiólogos están ocupados.
    2. Si es Código Verde, tranquiliza al paciente y ofrécele agendar cita con el médico disponible.
    3. Redacta la respuesta final que el paciente leerá en WhatsApp simulando que eres humano.
    """
    try:
        # Utilizando la nueva sintaxis de google-genai con el modelo detectado
        respuesta = cliente_gemini.models.generate_content(
            model=global_modelo_gemini,
            contents=prompt,
        )
        return respuesta.text
    except Exception as e:
        return f"[Error de redacción AI]: Revise su API Key. Detalles técnicos: {str(e)}"

def procesar_mensaje_whatsapp_rag(mensaje_paciente):
    # 1. Recuperación de Información (ClinicalBERT)
    protocolo = retriever.retrieve(mensaje_paciente)
    print(f"\n🧠 [Motor Interno - Protocolo Encontrado]: {protocolo}")
    
    # 2. Consulta a Base de Datos
    estado_db = f"Cardiología: {db.buscar_medico_disponible('Cardiología')} | General: {db.buscar_medico_disponible('Medicina General')}"
    
    # 3. Generación Aumentada (Gemini AI redacta basándose en los pasos 1 y 2)
    print(f"✨ [Gemini AI ({global_modelo_gemini}) Redactando respuesta empática...]")
    respuesta_final = generar_respuesta_gemini(mensaje_paciente, protocolo, estado_db)
    
    print("\n🤖 [Respuesta Final del Chatbot en WhatsApp]:")
    print(f"{respuesta_final}\n")


## Simulador de Flujo Interactivo Completo (Agendamiento + Triaje AI)

In [ ]:
import time

def agendar_cita_flujo():
    print("\n👤 Por favor, ingresa tu nombre completo:")
    nombre = input("> ")
    
    print("\n🪪 Digite su número de cedula:")
    cedula = input("> ")
    
    print("\n📞 Ingrese su número de teléfono:")
    telefono = input("> ")
    
    print("\n🩺 Selecciona una especialidad:")
    print("1️⃣ 🩺 Medicina General")
    print("2️⃣ ❤️ Cardiología")
    print("3️⃣ 🧠 Psicología")
    print("4️⃣ 🦴 Fisioterapia")
    esp_opcion = input("\nResponde con el número: ")
    
    especialidades = db.obtener_especialidades()
    if esp_opcion not in especialidades:
        print("\n❌ Opción inválida, volviendo al menú principal...")
        time.sleep(2)
        return
        
    esp_nombre = especialidades[esp_opcion]["nombre_esp"]
    doctores = especialidades[esp_opcion]["doctores"]
    
    print("\n👨‍⚕️ Ingrese el número del médico:")
    for idx, doc in enumerate(doctores):
        disp = "(Disponible)" if doc['disponible'] else "(Ocupado)"
        print(f"{idx+1}. {doc['nombre']} {disp}")
    
    doc_opcion = input("> ")
    try:
        doc_idx = int(doc_opcion) - 1
        medico_seleccionado = doctores[doc_idx]
        if not medico_seleccionado["disponible"]:
            print("\n❌ El médico seleccionado está ocupado. Volviendo al menú principal...")
            time.sleep(2)
            return
        medico_nombre = medico_seleccionado["nombre"]
    except:
        print("\n❌ Selección inválida. Volviendo al menú principal...")
        time.sleep(2)
        return

    print("\n📆 ¿Qué fecha prefieres para tu cita? (Ej: mañana, 25/03/2026)")
    fecha = input("> ")
    
    print("\n🕐 ¿A qué hora te gustaría? (Ej: 3pm, 14:30)")
    hora = input("> ")
    
    print("\n📝 Por favor, describe brevemente el motivo de tu consulta:")
    motivo = input("> ")
    
    print("\n📋 *Confirma tus datos: *")
    print(f"👤 Nombre: {nombre}")
    print(f"🪪 Cédula: {cedula}")
    print(f"🩺 Especialidad: {esp_nombre}")
    print(f"👨‍⚕️ Médico: {medico_nombre}")
    print(f"📆 Fecha: {fecha}")
    print(f"🕐 Hora: {hora}")
    print(f"📝 Motivo: {motivo}")
    
    confirmar = input("\n¿Confirmar? (sí/no): ").lower()
    if confirmar in ['si', 'sí', 's']:
        print(f"\n✅ *¡Cita agendada exitosamente!* Te esperamos el {fecha} a las {hora}.")
    else:
        print("\n❌ Cita cancelada.")
    time.sleep(3)

# ----------------- BUCLE PRINCIPAL -----------------
while True:
    print("\n" + "="*50)
    print("✨ *Bienvenido al Centro de Diálisis Margarita* ✨")
    print("Tu salud es nuestra prioridad. 🏥")
    print("\n¿En qué puedo ayudarte hoy?")
    print("*Opciones disponibles:*")
    print("1️⃣ 📅 Agendar Cita Completa")
    print("2️⃣ 🤖 Evaluación Médica (Triaje RAG AI)")
    print("3️⃣ ❓ Preguntas Frecuentes")
    print("4️⃣ 📞 Contacto")
    print("5️⃣ ❌ Salir del Simulador")
    
    opcion = input("\nResponde con el número de la opción: ")
    
    if opcion == '1':
        agendar_cita_flujo()
    elif opcion == '2':
        sintomas = input("\n👤 [Tú]: Por favor, describe tus síntomas médicos: ")
        print("\n⏳ Evaluando... (RAG Activo)")
        procesar_mensaje_whatsapp_rag(sintomas)
        time.sleep(3)
    elif opcion == '3':
        print("\n🤖 [Chatbot]: Nuestro horario de atención es de 8am a 6pm. Para más dudas, contacta a soporte.")
        time.sleep(2)
    elif opcion == '4':
        print("\n🤖 [Chatbot]: Puedes llamarnos al 01-8000-CLINICA o escribirnos a contacto@margarita.com")
        time.sleep(2)
    elif opcion == '5':
        print("\n🤖 [Chatbot]: Gracias por usar el sistema. ¡Cuídate mucho! 👋")
        break
    else:
        print("\n🤖 [Chatbot]: ⚠️ Opción no válida.")
        time.sleep(1)
